In [1]:
import sys
import pandas as pd
from pathlib import Path
import os
from glob import glob
from datetime import datetime
from pykrx import stock
from dateutil.relativedelta import *
from pandas.tseries.offsets import BDay
from DATA.stock_invest_function import *
from sqlalchemy import create_engine
import FinanceDataReader as fdr

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}


def get_cap(base_day):
    date = str(base_day.year) + str(base_day.month).zfill(2) + str(base_day.day).zfill(2)
    limit_num = 1
    while limit_num < 10:
        market_info = stock.get_market_cap(date)
        if market_info['시가총액'].sum() != 0:
            break
        else:
            prev_date = base_day + relativedelta(days=-limit_num)
            date = str(prev_date.year) + str(prev_date.month).zfill(2) + str(prev_date.day).zfill(2)
            limit_num += 1

    market_info.reset_index(inplace=True)
    market_info.rename(columns={'단축코드': '종목코드', '상장주식수': 'Stocks'}, inplace=True)
    return market_info

In [2]:
# ===== 날짜 리스트 생성 =====
start_date = "2025-08-01"
end_date = "2025-09-08"
biz_days = pd.date_range(start=start_date, end=end_date, freq=BDay())  # 휴일 제외 영업일

# ===== 날짜별 시가총액 수집 =====
all_data = []

for day in biz_days:
    print(f"📅 {day.date()} 시가총액 데이터 수집중...")
    cap_df = get_cap(day)
    cap_df['date'] = day.strftime('%Y-%m-%d')
    all_data.append(cap_df)

# ===== 하나의 DataFrame으로 병합 =====
result_df = pd.concat(all_data, ignore_index=True)



📅 2025-08-01 시가총액 데이터 수집중...
📅 2025-08-04 시가총액 데이터 수집중...
📅 2025-08-05 시가총액 데이터 수집중...
📅 2025-08-06 시가총액 데이터 수집중...
📅 2025-08-07 시가총액 데이터 수집중...
📅 2025-08-08 시가총액 데이터 수집중...
📅 2025-08-11 시가총액 데이터 수집중...
📅 2025-08-12 시가총액 데이터 수집중...
📅 2025-08-13 시가총액 데이터 수집중...
📅 2025-08-14 시가총액 데이터 수집중...
📅 2025-08-15 시가총액 데이터 수집중...
📅 2025-08-18 시가총액 데이터 수집중...
📅 2025-08-19 시가총액 데이터 수집중...
📅 2025-08-20 시가총액 데이터 수집중...
📅 2025-08-21 시가총액 데이터 수집중...
📅 2025-08-22 시가총액 데이터 수집중...
📅 2025-08-25 시가총액 데이터 수집중...
📅 2025-08-26 시가총액 데이터 수집중...
📅 2025-08-27 시가총액 데이터 수집중...
📅 2025-08-28 시가총액 데이터 수집중...
📅 2025-08-29 시가총액 데이터 수집중...
📅 2025-09-01 시가총액 데이터 수집중...
📅 2025-09-02 시가총액 데이터 수집중...
📅 2025-09-03 시가총액 데이터 수집중...
📅 2025-09-04 시가총액 데이터 수집중...
📅 2025-09-05 시가총액 데이터 수집중...
📅 2025-09-08 시가총액 데이터 수집중...


In [3]:
# ✅ 컬럼 이름 변경
result_df.rename(columns={'티커': 'ticker', 'Stocks': '유통주식수'}, inplace=True)

# ✅ ticker 값 앞에 'A' 붙이기
result_df['ticker'] = 'A' + result_df['ticker'].astype(str)

# ✅ long format 으로 변환
long_df = result_df.melt(
    id_vars=['date', 'ticker'],    # 고정할 컬럼
    var_name='indicator',          # 측정 대상의 이름을 넣을 컬럼
    value_name='value'             # 측정 값
)

# ✅ DB 연결 엔진 생성
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
)

# ✅ long_df 를 DB로 업로드
# if_exists 옵션:
# 'replace' -> 기존 테이블 삭제 후 재생성
# 'append' -> 테이블이 있으면 데이터만 추가
long_df.to_sql(
    name='ks_listed_company_daily_marketcap',
    con=engine,
    if_exists='append',  # 또는 'replace' 가능
    index=False
)

print("✅ ks_listed_company_daily_marketcap 테이블로 업로드 완료!")

✅ ks_listed_company_daily_marketcap 테이블로 업로드 완료!


In [7]:
long_df['indicator'].unique().tolist()

['종가', '시가총액', '거래량', '거래대금', '유통주식수']

In [5]:
long_df[long_df['ticker'] == 'A005930']

,date,ticker,indicator,value
0,2025-08-01,A005930,종가,68900
2876,2025-08-04,A005930,종가,69700
5752,2025-08-05,A005930,종가,69900
8628,2025-08-06,A005930,종가,68800
11505,2025-08-07,A005930,종가,70500
...,...,...,...,...
374315,2025-09-02,A005930,유통주식수,5919637922
377196,2025-09-03,A005930,유통주식수,5919637922
380077,2025-09-04,A005930,유통주식수,5919637922
382958,2025-09-05,A005930,유통주식수,5919637922
